# الفصل 24: التدريب المسبق المستمر: CPT

> تخيّل أنك تملك نموذجًا عامًا ممتازًا (مثل Llama 3 8B) لكنه ضعيف في المجال الطبي العربي أو القانون السعودي أو الكود بلغة برمجة نادرة.

- **الجزء:** 6 — إعادة استخدام النماذج
- **الفصل:** 24
- **المستوى:** متقدم
- **زمن القراءة:** 15 دقيقة
- **المتطلبات:** الفصل 23: خط إنتاج ما بعد التدريب
- **الوسوم:** CPT، DAPT، Catastrophic Forgetting، EWC

## كيف تشغّل هذا الدفتر؟

1. **Google Colab:** ارفع الملف عبر `File → Upload notebook`، ثم `Runtime → Run all`.
2. **Kaggle:** أنشئ Notebook ثم `File → Import notebook`، ثم `Run All`.
3. شغّل خلية **إعداد البيئة** أولًا؛ ستفحص الحزم وتثبّت الناقص تلقائيًا.
4. للتدريب أو النماذج الكبيرة: Colab `Runtime → Change runtime type → T4 GPU`، وKaggle `Settings → Accelerator → GPU`.


<div dir="rtl" style="margin:2px 0 18px;line-height:2.4;">
<b>افتح هذا الدفتر على منصة مجانية:</b><br>
<a href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#F9AB00;color:#111111;">Google Colab</a> <a href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#20BEFF;color:#FFFFFF;">Kaggle</a> <a href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#579ACA;color:#FFFFFF;">Binder</a> <a href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#6C5CE7;color:#FFFFFF;">Gradient</a> <a href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#37474F;color:#FFFFFF;">nbviewer</a> <a href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" style="display:inline-block;margin:3px;padding:6px 14px;border-radius:999px;text-decoration:none;font-weight:600;font-size:13px;background:#24292F;color:#FFFFFF;">GitHub</a>
</div>


## ملاحظات التشغيل

- **الحزم:** torch, transformers
- **GPU:** يستخدم هذا الدفتر PyTorch أو نماذج كبيرة؛ يُفضّل GPU. يمكنك البدء بنماذج صغيرة على CPU، لكن التدريب الكامل سيكون بطيئًا.
- **بيانات/نماذج خارجية:** بعض الخلايا تتوقع ملفات أو نماذج محمّلة مسبقًا. عدّل المسارات أو حمّل البيانات قبل التشغيل.
- **خلايا ثقيلة:** قد تستغرق خلايا التدريب/التحميل وقتًا طويلًا. شغّلها خلية بخية، وقلّل `batch_size` أو استخدم نموذجًا أصغر عند الحاجة.
- **مقاطع توضيحية:** بعض الشيفرة تحتوي `...` أو `NotImplementedError`؛ هذا مقصود لبيان البنية وليس خطأً في الدفتر.


<style>
/* نص عربي من اليمين إلى اليسار، مع إبقاء الشيفرة من اليسار */
.rendered_html, .jp-RenderedHTMLCommon, .text_cell_render, .markdown-body {
  direction: rtl;
  text-align: right;
}
.rendered_html pre, .rendered_html code,
.jp-RenderedHTMLCommon pre, .jp-RenderedHTMLCommon code,
.text_cell_render pre, .text_cell_render code,
.markdown-body pre, .markdown-body code {
  direction: ltr;
  text-align: left;
  unicode-bidi: embed;
}
.rendered_html table, .jp-RenderedHTMLCommon table, .text_cell_render table {
  direction: rtl;
}
</style>


In [ ]:
# === إعداد البيئة ===
# يفحص هذا القسم الحزم المطلوبة ويحاول تثبيت الناقص منها تلقائيًا.
# شغّل هذه الخلية أولًا، ثم بقية خلايا الدفتر.
import importlib.util
import subprocess
import sys

print("Python:", sys.version.split()[0])

# (اسم وحدة الاستيراد، اسم حزمة pip)
REQUIRED = [
    ("torch", "torch"),
    ("transformers", "transformers")
]
AUTO_INSTALL = {"PIL", "bs4", "datasets", "faiss", "fasttext", "librosa", "networkx", "opentelemetry", "peft", "prometheus_client", "pydantic", "rank_bm25", "requests", "sentence_transformers", "sentencepiece", "soundfile", "tokenizers", "trafilatura", "transformers", "wandb", "yaml"}


def has_module(name):
    try:
        return importlib.util.find_spec(name) is not None
    except Exception:
        return False


if not REQUIRED:
    print("لا توجد مكتبات خارجية مطلوبة؛ يكفي Python القياسي.")
else:
    missing = [(module, package) for module, package in REQUIRED if not has_module(module)]
    auto_specs = []
    manual_specs = []
    for module, package in missing:
        specs = package.split()
        if module in AUTO_INSTALL:
            auto_specs.extend(specs)
        else:
            manual_specs.extend(specs)
    auto_specs = list(dict.fromkeys(auto_specs))
    manual_specs = list(dict.fromkeys(manual_specs))

    if auto_specs:
        print("⏳ تثبيت تلقائي:", " ".join(auto_specs))
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *auto_specs])
            print("✅ تم التثبيت.")
        except Exception as exc:
            print("⚠️ تعذّر التثبيت التلقائي:", exc)
            print("شغّل يدويًا: %pip install", " ".join(auto_specs))
    if manual_specs:
        print("⚠️ حزم تحتاج تثبيتًا يدويًا:", " ".join(manual_specs))
        print("شغّل: %pip install", " ".join(manual_specs))
    if not auto_specs and not manual_specs:
        print("✅ كل الحزم المطلوبة متاحة.")

    print("\nالحالة النهائية:")
    for module, package in REQUIRED:
        print(("✅" if has_module(module) else "⚠️"), package)


<div class="notebook-actions" dir="rtl"><span class="notebook-actions__label">شغّل الدفتر:</span><a class="nb-action nb-action--colab" href="https://colab.research.google.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="افتح في Google Colab — GPU مجاني">Google Colab</a><a class="nb-action nb-action--kaggle" href="https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2FAhmedAlmaghz%2FMLLBook%2Fblob%2Fmain%2Fnotebooks%2Fpart-06-model-reuse%2Fch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="افتح في Kaggle — GPU/TPU مجاني">Kaggle</a><a class="nb-action nb-action--binder" href="https://mybinder.org/v2/gh/AhmedAlmaghz/MLLBook/main?labpath=notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="افتح في Binder — بيئة Jupyter مجانية">Binder</a><a class="nb-action nb-action--gradient" href="https://console.paperspace.com/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="افتح في Paperspace Gradient — GPU مجاني محدود">Gradient</a><a class="nb-action nb-action--nbviewer" href="https://nbviewer.org/github/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="اعرض الدفتر دون تنفيذ">nbviewer</a><a class="nb-action nb-action--github" href="https://github.com/AhmedAlmaghz/MLLBook/blob/main/notebooks/part-06-model-reuse/ch24-continued-pretraining.ipynb" target="_blank" rel="noopener" title="اعرض الملف المصدر على GitHub">GitHub</a></div>

> **ملخص الفصل**
>
> تخيّل أنك تملك نموذجًا عامًا ممتازًا (مثل Llama 3 8B) لكنه ضعيف في المجال الطبي العربي أو القانون السعودي أو الكود بلغة برمجة نادرة.

> **بيانات الفصل**
>
> - **الجزء:** 6 — إعادة استخدام النماذج
> - **المستوى:** متقدم
> - **زمن القراءة:** 15 دقيقة
> - **المتطلبات:** الفصل 23: خط إنتاج ما بعد التدريب


## مخرجات التعلم

بنهاية هذا الفصل ستكون قادرًا على:

- فهم متى تحتاج Continued Pretraining بدل SFT مباشرة؟
- فهم التكيّف مع المجال: Domain-Adaptive Pretraining (DAPT)
- فهم مشكلة النسيان الكارثي (Catastrophic Forgetting)


## 1. لماذا يهمّك هذا الموضوع؟

🟢 طبقة الحدس
تخيّل أنك تملك نموذجًا عامًا ممتازًا (مثل Llama 3 8B) لكنه ضعيف في المجال الطبي العربي أو القانون السعودي أو الكود بلغة برمجة نادرة. لديك خياران: (1) تدريب نموذج جديد من الصفر على بيانات هذا المجال — مكلف جدًا ويتطلب مليارات الرموز وأسابيع من GPU، أو (2) متابعة تدريب النموذج الموجود على بيانات المجال الجديد فقط لبضعة مليارات رمز إضافية. الخيار الثاني هو Continued Pretraining (CPT)، وهو غالبًا أرخص بـ 100-1000 مرة من البدء من الصفر، لأنك تستغل كل المعرفة اللغوية والعالمية التي اكتسبها النموذج فعلًا، وتُضيف فقط الطبقة المتخصصة المفقودة.


## 2. الخريطة الذهنية


```text
دورة حياة نموذج LLM الكاملة (مُحدَّثة بإضافة CPT)

Pretraining من الصفر (الجزء 5)
        │
        ▼
النموذج الأساسي العام (Base Model)
        │
        ├──→ [مسار مباشر] SFT + RLHF/DPO (الجزء 7) → نموذج محادثة عام
        │
        └──→ [مسار CPT] Continued Pretraining على بيانات مجال مُحدَّد
                   │
                   ▼
            نموذج أساسي مُتخصِّص بالمجال
                   │
                   ▼
            SFT + RLHF/DPO مُتخصِّص بالمجال
                   │
                   ▼
            نموذج محادثة مُتخصِّص (مثال: نموذج طبي عربي)
```


## 3. الشرح المتدرج

### 3.1 متى تحتاج Continued Pretraining بدل SFT مباشرة؟

🟡 طبقة الآلية

السؤال الحاسم: هل الفجوة بين النموذج الحالي ومهمتك المستهدفة هي فجوة معرفية (النموذج لا يعرف المصطلحات/الحقائق/الأسلوب اللغوي للمجال) أم فجوة سلوكية (النموذج يعرف المعرفة لكن لا "يتبع التعليمات" بالشكل المطلوب)؟

| نوع الفجوة | الحل المناسب | مثال |
|---|---|---|
| فجوة معرفية عميقة (مفردات ومفاهيم جديدة كليًا) | Continued Pretraining على كمّ كبير من نصوص المجال الخام | نموذج عام → نموذج متخصص في الكيمياء الحيوية |
| فجوة لغوية (لغة/لهجة ضعيفة التمثيل في بيانات التدريب الأصلية) | Continued Pretraining على نصوص باللغة/اللهجة المستهدفة | نموذج إنجليزي بالأساس → نموذج عربي فصيح وقوي باللهجات |
| فجوة سلوكية (يعرف المعلومة لكن لا يُجيب بالتنسيق المطلوب) | SFT فقط (الجزء 7) كافٍ، بدون CPT | نموذج عام → مساعد يُجيب بصيغة JSON محدّدة |
| فجوة في التفضيلات (الإجابات صحيحة لكن الأسلوب غير مرغوب) | DPO/RLHF فقط (الجزء 7) | نموذج يُعطي إجابات طويلة جدًا → نموذج موجز ومباشر |

> ⚠️ خطأ شائع جوهري: محاولة "تعليم" النموذج معرفة جديدة كليًا (مثل: قانون جديد صدر بعد تاريخ تدريب النموذج) عبر SFT فقط بأمثلة قليلة (مئات أو آلاف الأمثلة). SFT ممتاز لتعليم السلوك والتنسيق، لكنه ضعيف جدًا في حقن معرفة واقعية جديدة بشكل موثوق ومُعمَّم (النموذج قد "يحفظ" الأمثلة القليلة حرفيًا دون أن "يستوعب" المعرفة الأساسية خلفها) — هذا يتطلب التعرّض لكمّ أكبر بكثير من النص الخام عبر CPT.

### 3.2 التكيّف مع المجال: Domain-Adaptive Pretraining (DAPT)

🟡 طبقة الآلية

ورقة Don't Stop Pretraining (Gururangan et al., 2020) أثبتت مبكرًا أن إضافة مرحلة تدريب مستمر على نصوص المجال الخام (حتى بكميات متواضعة نسبيًا: مئات الملايين إلى بضعة مليارات رمز) تُحسِّن الأداء النهائي على مهام المجال بشكل ملموس، أكثر من الاكتفاء بـ SFT مباشرة على نموذج عام.


In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup

class ContinuedPretrainingPipeline:
    """
    خط أنابيب كامل للتدريب المسبق المستمر (CPT) على نموذج أساسي موجود،
    مع إدارة دقيقة لمعدل التعلّم (LR Re-warming) ومزج البيانات لتفادي
    النسيان الكارثي (القسم 3.3 أدناه).
    """
    def __init__(self, base_model_name: str, domain_corpus_path: str,
                 general_corpus_path: str = None, device: str = "cuda"):
        self.model = AutoModelForCausalLM.from_pretrained(
            base_model_name, torch_dtype=torch.bfloat16
        ).to(device)
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        self.device = device
        self.domain_corpus_path = domain_corpus_path
        self.general_corpus_path = general_corpus_path  # لمزج "تذكير" (replay)

    def build_mixed_dataset(self, domain_replay_ratio: float = 0.7):
        """
        بناء مجموعة بيانات مُمزوجة: نسبة كبيرة من بيانات المجال الجديد
        + نسبة أصغر من بيانات عامة "تذكيرية" (Replay Buffer) لتثبيت
        القدرات العامة الأصلية ومنع نسيانها (راجع القسم 3.3).
        """
        domain_texts = self._load_corpus(self.domain_corpus_path)
        general_texts = (
            self._load_corpus(self.general_corpus_path)
            if self.general_corpus_path else []
        )

        n_domain = int(len(domain_texts))
        n_general_needed = int(
            n_domain * (1 - domain_replay_ratio) / domain_replay_ratio
        )
        general_sample = general_texts[:n_general_needed] if general_texts else []

        mixed = domain_texts + general_sample
        import random
        random.shuffle(mixed)  # خلط عشوائي - مهم جدًا لمنع تجميع نوع واحد
        return mixed

    def _load_corpus(self, path):
        with open(path, encoding="utf-8") as f:
            return [line.strip() for line in f if line.strip()]

    def get_cpt_optimizer_and_scheduler(self, train_steps: int,
                                         peak_lr: float = 5e-6,
                                         warmup_ratio: float = 0.03):
        """
        نقطة حاسمة في CPT: معدل التعلّم الذروي (peak_lr) يجب أن يكون
        أصغر بكثير من peak_lr الأصلي للتدريب المسبق الأول (عادة أقل
        بـ 5-10 مرات)، لأن النموذج بالفعل في منطقة جيدة من فضاء
        الأوزان ولا نريد "تدميرها" بتحديثات كبيرة. يُنصح أيضًا
        بـ warmup قصير (LR Re-warming) بدل البدء المباشر من peak_lr.
        """
        optimizer = torch.optim.AdamW(
            self.model.parameters(), lr=peak_lr, weight_decay=0.01,
            betas=(0.9, 0.95)  # betas معدّلة شائعة في تدريب LLM الحديث
        )
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(train_steps * warmup_ratio),
            num_training_steps=train_steps,
        )
        return optimizer, scheduler

    def train(self, mixed_dataset, num_epochs: int = 1, batch_size: int = 4,
               max_seq_len: int = 2048, gradient_accumulation_steps: int = 8):
        """حلقة تدريب CPT كاملة مع كل الاحتياطات اللازمة."""
        tokenized = [
            self.tokenizer(t, truncation=True, max_length=max_seq_len,
                            return_tensors="pt")["input_ids"][0]
            for t in mixed_dataset
        ]
        total_steps = (len(tokenized) // batch_size) * num_epochs
        optimizer, scheduler = self.get_cpt_optimizer_and_scheduler(total_steps)

        self.model.train()
        step = 0
        for epoch in range(num_epochs):
            for i in range(0, len(tokenized) - batch_size, batch_size):
                batch = tokenized[i:i + batch_size]
                batch_padded = torch.nn.utils.rnn.pad_sequence(
                    batch, batch_first=True, padding_value=self.tokenizer.pad_token_id
                ).to(self.device)

                outputs = self.model(input_ids=batch_padded, labels=batch_padded)
                loss = outputs.loss / gradient_accumulation_steps
                loss.backward()

                if (step + 1) % gradient_accumulation_steps == 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                if step % 50 == 0:
                    print(f"خطوة {step}: الخسارة = {outputs.loss.item():.4f}, "
                          f"LR = {scheduler.get_last_lr()[0]:.2e}")
                step += 1

        return self.model


### 3.3 مشكلة النسيان الكارثي (Catastrophic Forgetting)

🔴 طبقة التعمّق

أكبر خطر في CPT: كلما دربت النموذج أكثر على بيانات المجال الجديد، كلما نسي قدراته العامة الأصلية (محادثة عامة، برمجة، استدلال منطقي عام) — ظاهرة معروفة في التعلّم المستمر (Continual Learning) باسم Catastrophic Forgetting.


In [ ]:
def measure_catastrophic_forgetting(
    original_model, cpt_model, general_benchmark_suite: dict,
    domain_benchmark_suite: dict,
) -> dict:
    """
    قياس كمي لمدى النسيان الكارثي: مقارنة أداء النموذج قبل وبعد CPT
    على مجموعتين من المعايير: عامة (يجب ألا تنخفض كثيرًا) ومجال
    مُتخصِّص (يجب أن ترتفع بوضوح - وإلا فإن CPT لم يحقق هدفه أصلًا).
    """
    results = {"general": {}, "domain": {}}

    for benchmark_name, benchmark_data in general_benchmark_suite.items():
        original_score = evaluate_on_benchmark(original_model, benchmark_data)
        cpt_score = evaluate_on_benchmark(cpt_model, benchmark_data)
        forgetting = original_score - cpt_score  # رقم موجب = نسيان حقيقي
        results["general"][benchmark_name] = {
            "before": original_score, "after": cpt_score,
            "forgetting": round(forgetting, 4),
            "status": "⚠️ نسيان ملحوظ" if forgetting > 0.05 else "✅ مستقر",
        }

    for benchmark_name, benchmark_data in domain_benchmark_suite.items():
        original_score = evaluate_on_benchmark(original_model, benchmark_data)
        cpt_score = evaluate_on_benchmark(cpt_model, benchmark_data)
        improvement = cpt_score - original_score  # رقم موجب = تحسّن مرغوب
        results["domain"][benchmark_name] = {
            "before": original_score, "after": cpt_score,
            "improvement": round(improvement, 4),
            "status": "✅ تحسّن جيد" if improvement > 0.05 else "⚠️ تحسّن ضعيف",
        }

    return results

def evaluate_on_benchmark(model, benchmark_data) -> float:
    # دالة تقييم مرجعية - راجع الجزء التاسع (فصل 41-42) للتفاصيل الكاملة
    raise NotImplementedError("يُطبَّق باستخدام أدوات التقييم من الفصل 41-42")


الاستراتيجيات الثلاث الرئيسية لمكافحة النسيان الكارثي:


```text
1. Experience Replay (الأكثر شيوعًا وفعالية)
   مزج نسبة من البيانات العامة الأصلية ضمن بيانات CPT
   (كما في build_mixed_dataset أعلاه - عادة 10-30% بيانات عامة)

2. Lower Learning Rate + Fewer Epochs
   استخدام LR أصغر بكثير من التدريب الأصلي + تمرير واحد (epoch=1)
   فقط عبر بيانات CPT بدل عدة تمريرات

3. Elastic Weight Consolidation (EWC) - أكثر تعقيدًا
   إضافة عقوبة في دالة الخسارة تُثبّت الأوزان "المهمة" للمهام
   القديمة (مُقدَّرة عبر Fisher Information Matrix) وتمنعها من
   الانحراف الكبير أثناء تدريب CPT
```


In [ ]:
class ElasticWeightConsolidation:
    """
    تطبيق EWC لتقليل النسيان الكارثي: نُقدِّر "أهمية" كل وزن بالنسبة
    للمهمة الأصلية عبر مصفوفة معلومات فيشر (تقريب قطري)، ثم نعاقب
    أي انحراف كبير عن القيمة الأصلية للأوزان المهمة أثناء CPT.
    """
    def __init__(self, model, ewc_lambda: float = 1000.0):
        self.model = model
        self.ewc_lambda = ewc_lambda
        self.fisher_info = {}
        self.original_params = {}

    def compute_fisher_information(self, general_dataloader):
        """تقدير مصفوفة فيشر القطرية باستخدام بيانات المهمة الأصلية."""
        self.model.eval()
        fisher = {n: torch.zeros_like(p) for n, p in self.model.named_parameters()
                   if p.requires_grad}

        for batch in general_dataloader:
            self.model.zero_grad()
            outputs = self.model(**batch, labels=batch["input_ids"])
            outputs.loss.backward()
            for n, p in self.model.named_parameters():
                if p.requires_grad and p.grad is not None:
                    fisher[n] += p.grad.data ** 2 / len(general_dataloader)

        self.fisher_info = fisher
        self.original_params = {
            n: p.clone().detach() for n, p in self.model.named_parameters()
        }

    def ewc_penalty(self) -> torch.Tensor:
        """حساب عقوبة EWC لإضافتها لخسارة CPT الأساسية."""
        penalty = torch.tensor(0.0, device=next(self.model.parameters()).device)
        for n, p in self.model.named_parameters():
            if n in self.fisher_info:
                penalty += (
                    self.fisher_info[n] * (p - self.original_params[n]) ** 2
                ).sum()
        return self.ewc_lambda * penalty

    def combined_loss(self, cpt_task_loss: torch.Tensor) -> torch.Tensor:
        return cpt_task_loss + self.ewc_penalty()


### 3.4 إعادة الإحماء (LR Re-warming) ومزج البيانات

📐 منحنى معدل التعلّم الموصى به لـ CPT


```text
LR
│
│         ╱‾‾‾╲
│        ╱     ╲___
│       ╱           ╲___
│      ╱                 ╲____
│_____╱                        ╲___________
└──────────────────────────────────────────→ خطوات التدريب
  Warmup قصير   الذروة (أقل من      Cosine Decay تدريجي
  (1-3%)        الأصلي بـ 5-10×)    حتى peak_lr/10 تقريبًا
```


## 4. مثال محلول خطوة بخطوة

السيناريو: لدينا Llama 3 8B عام، نريد تخصيصه للمجال الطبي العربي.


In [ ]:
# الخطوة 1: تجهيز بيانات المجال (نصوص طبية عربية خام - أبحاث، كتب، تقارير)
pipeline = ContinuedPretrainingPipeline(
    base_model_name="meta-llama/Meta-Llama-3-8B",
    domain_corpus_path="arabic_medical_corpus.txt",  # ~2 مليار رمز
    general_corpus_path="arabic_general_corpus_sample.txt",  # للـ Replay
)

# الخطوة 2: بناء مجموعة بيانات ممزوجة (80% طبي، 20% عام للتذكير)
mixed_data = pipeline.build_mixed_dataset(domain_replay_ratio=0.8)

# الخطوة 3: التدريب بمعدل تعلّم محافظ جدًا وتمرير واحد فقط
trained_model = pipeline.train(
    mixed_dataset=mixed_data, num_epochs=1, batch_size=4,
    max_seq_len=4096, gradient_accumulation_steps=16,
)

# الخطوة 4: قياس النسيان الكارثي قبل المتابعة لمرحلة SFT
forgetting_report = measure_catastrophic_forgetting(
    original_model=pipeline.model,  # يُفترض نسخة أصلية محفوظة مسبقًا
    cpt_model=trained_model,
    general_benchmark_suite={"MMLU_general": ..., "HellaSwag": ...},
    domain_benchmark_suite={"Arabic_Medical_QA": ..., "MedMCQA_Arabic": ...},
)
# توقع: ارتفاع واضح في Arabic_Medical_QA، انخفاض طفيف جدًا (<3%) في MMLU_general


## 5. نصائح وحيل

احتفظ دائمًا بنسخة من النموذج الأصلي قبل CPT (checkpoint منفصل) — لا بديل عن المقارنة المباشرة لقياس النسيان.
حجم بيانات CPT الموصى به كقاعدة أولية: 1-5% من حجم بيانات التدريب المسبق الأصلي (مثلًا، إذا دُرِّب النموذج الأصلي على 15 تريليون رمز، فـ 10-50 مليار رمز من بيانات المجال كافية غالبًا لنقلة نوعية).
راقب خسارة التدريب (Training Loss) بعناية: قفزة حادة في بداية CPT (أعلى من الخسارة النهائية للتدريب الأصلي) أمر متوقع وطبيعي (توزيع البيانات الجديد مختلف)، لكن إذا لم تنخفض الخسارة خلال الـ epoch الأولى فهناك مشكلة في LR أو جودة البيانات.


## 6. أخطاء شائعة ومصائد

استخدام نفس peak_lr للتدريب المسبق الأصلي في CPT — يؤدي غالبًا لتدمير جزئي لقدرات النموذج العامة (نسيان كارثي حاد).
تجاهل تنظيف بيانات المجال بنفس صرامة تنظيف بيانات التدريب المسبق الأصلي (الفصل 18) — بيانات مجال "خام" وغير مُنقَّاة قد تُدخِل ضوضاء وتُضعِف النتيجة النهائية بدل تحسينها.
عدم تضمين أي بيانات Replay عامة إطلاقًا، معتقدين أن "التركيز الكامل" على المجال أفضل — هذا يُسرِّع النسيان الكارثي بشكل كبير.


## 7. جدول مقارنة

| المعيار | Continued Pretraining | SFT مباشرة فقط | التدريب من الصفر |
|---|---|---|---|
| التكلفة الحاسوبية | متوسطة (مليارات رموز) | منخفضة جدًا (آلاف-ملايين أمثلة) | ضخمة جدًا (تريليونات رموز) |
| فعالية حقن معرفة جديدة | عالية جدًا | ضعيفة | عالية جدًا لكن بتكلفة هائلة |
| خطر النسيان الكارثي | متوسط (قابل للإدارة) | منخفض | غير وارد (لا يوجد "أصل" لينسى) |
| يتطلب بنية تحتية ضخمة؟ | لا (عقدة GPU واحدة كافية غالبًا) | لا | نعم (مجموعات GPU كبيرة) |
| الاستخدام الأمثل | تخصيص لغوي/مجالي عميق | تعليم تنسيق/سلوك محدد | بناء نموذج أساسي جديد كليًا |


## 8. ملخص أفضل الممارسات

✅ استخدم CPT عندما تكون الفجوة معرفية/لغوية عميقة، وSFT فقط عندما تكون الفجوة سلوكية/تنسيقية.
✅ امزج دائمًا نسبة بيانات عامة (Replay) ضمن بيانات CPT (10-30% كقاعدة ابتدائية).
✅ استخدم peak_lr أصغر بـ 5-10× من التدريب الأصلي، مع warmup قصير.
✅ قِس النسيان الكارثي كميًا على معايير عامة قبل وبعد، لا تعتمد فقط على تحسّن أداء المجال.
✅ تمريرة واحدة (epoch=1) على بيانات CPT عادة كافية وأكثر أمانًا من عدة تمريرات.


## 9. أسئلة مقابلات

**سؤال: متى تُفضِّل Continued Pretraining على SFT لحل مشكلة "النموذج لا يعرف شيئًا عن مجالي"؟**

**إجابة نموذجية:** SFT ممتاز لتعليم النموذج كيف يتصرف (تنسيق، أسلوب، اتباع تعليمات) باستخدام عدد محدود من الأمثلة عالية الجودة، لكنه ضعيف في حقن معرفة واقعية جديدة بكمية كبيرة بشكل مُعمَّم وموثوق — لأن عدد الأمثلة في SFT (عادة آلاف إلى عشرات الآلاف) ضئيل جدًا مقارنة بالتعرّض المطلوب لتعلّم مفردات ومفاهيم مجال كامل. Continued Pretraining، باستخدام نفس هدف Next-Token-Prediction للتدريب المسبق الأصلي لكن على نصوص خام من المجال المستهدف وبكميات أكبر بكثير (مليارات الرموز)، يسمح للنموذج باستيعاب هذه المعرفة الجديدة بعمق وتعميمها، تمامًا كما استوعب معرفته العامة الأصلية. المسار الأمثل عادة هو: CPT أولاً لحقن المعرفة، ثم SFT على بيانات المجال لتعليم التنسيق والسلوك المطلوبين فوق هذه المعرفة الجديدة.

**سؤال: اشرح كيف تعمل Elastic Weight Consolidation (EWC) في مكافحة النسيان الكارثي، وما هو القيد العملي الأكبر لاستخدامها مع نماذج ضخمة؟**

**إجابة نموذجية:** EWC تُقدِّر "أهمية" كل وزن بالنسبة للمهمة/المعرفة الأصلية عبر مصفوفة معلومات فيشر (تقريبًا: مربع التدرج لكل وزن محسوبًا على بيانات المهمة الأصلية) — الأوزان ذات القيمة العالية في مصفوفة فيشر تُعتبر "حرجة" لأداء المهمة القديمة. أثناء CPT، تُضاف عقوبة تربيعية (شبيهة بـ L2 لكن مُرجَّحة بأهمية كل وزن) تمنع هذه الأوزان الحرجة من الانحراف الكبير عن قيمتها الأصلية، بينما تُترك الأوزان الأقل أهمية حرة للتكيّف مع المهمة الجديدة. القيد العملي الأكبر: حساب وتخزين مصفوفة فيشر القطرية (حتى في تقريبها الأبسط) يتطلب ذاكرة إضافية تساوي حجم النموذج بالكامل (قيمة فيشر واحدة لكل وزن) بالإضافة لنسخة من الأوزان الأصلية — وهذا عبء ذاكرة كبير جدًا على نماذج بمليارات المعاملات، مما يجعل Experience Replay (مزج بيانات عامة) الحل الأكثر عملية وانتشارًا في الصناعة رغم بساطته الشديدة مقارنة بـ EWC.


## 10. تمرين عملي

باستخدام ContinuedPretrainingPipeline، صمّم تجربة على نموذج صغير (مثل GPT-2 124M أو TinyLlama) لتكييفه مع نصوص برمجية بلغة نادرة نسبيًا (مثل Rust أو Zig). طبّق measure_catastrophic_forgetting بمقارنة أداء النموذج على اختبار برمجة عام (Python) قبل وبعد CPT، مع نسب Replay مختلفة (0%, 20%, 50%)، وارسم العلاقة بين نسبة Replay ومقدار النسيان مقابل تحسّن أداء المجال الجديد.


## 11. مراجع للتعمّق

- Gururangan et al. (2020): Don't Stop Pretraining: Adapt Language Models to Domains and Tasks
- Kirkpatrick et al. (2017): Overcoming Catastrophic Forgetting in Neural Networks (EWC)
- Ibrahim et al. (2024): Simple and Scalable Strategies to Continually Pre-train Large Language Models
- Parmar et al. (2024, NVIDIA): Reuse, Don't Retrain: A Recipe for Continued Pretraining of Language Models


## 🧪 مساحة تجربة

استخدم الخلية التالية لتجربة أفكار الفصل أو حلّ التمرين. عدّل القيم وأعد تشغيل الخلية.


In [ ]:
# 🧪 مساحة تجربة
# اكتب هنا حلّ التمرين أو أي تجربة إضافية.


---
📘 **المصدر:** كتاب *إتقان المحوّلات والنماذج اللغوية الكبيرة* — إعادة استخدام النماذج / الفصل 24: التدريب المسبق المستمر: CPT.

هذا الدفتر مولّد آليًا من ملف الفصل؛ لا تعدّله يدويًا، بل عدّل المصدر ثم أعد البناء.
